# 02 — Retention Cohorts

**Question:** are newer registration cohorts retaining better or worse than older ones, and
when subscribers don't renew, is revenue lost to voluntary cancellation or to silent lapse?
Those are different problems with different fixes — a voluntary cancel is a decision the
business had a chance to intervene on; a silent lapse often has no warning signal at all.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

sys.path.append(str(Path.cwd().parent))
from src import cohorts, plotting

plotting.set_style()

interim = Path("../data/interim")
periods = pd.read_parquet(interim / "periods.parquet")
members_cohort = pd.read_parquet(interim / "members_cohort.parquet")

## Retention curve, all cohorts (heatmap)
Sequential color = magnitude. Read down a column to compare cohorts at the same tenure; read across a row to see one cohort's own decay curve.

In [ ]:
curve = cohorts.cohort_retention_curve(periods, members_cohort, max_cycle=12)
pivot = curve.pivot(index="cohort_month", columns="cycle", values="retention_rate").sort_index()

fig, ax = plotting.plot_cohort_heatmap(
    pivot, "Retention rate by cohort and subscription cycle",
    save_path="../figures/02_cohort_heatmap.png",
)

**How to read this:** a column that gets systematically lighter toward the bottom rows (the most recent cohorts) is the "newer cohorts churn faster" signal. A column that gets darker toward the bottom is the opposite. Compare within a column, not across the diagonal — cohorts at the bottom haven't had time to reach the columns on the right yet (see the reachable-horizon filter below).

## Reachable-horizon filter
Recent cohorts haven't had enough elapsed time to reach cycle 6+ — comparing them there would blame churn for what's actually just missing follow-up time.

In [ ]:
reachable = cohorts.cohort_max_reachable_cycle(periods, members_cohort)
COMPARE_CYCLE = 6
eligible_cohorts = reachable[reachable >= COMPARE_CYCLE].index

print(f"{len(eligible_cohorts)} of {len(reachable)} cohorts have reached cycle {COMPARE_CYCLE} and are eligible for comparison.")
print("Most recent 6 cohorts excluded as too young:", sorted(set(reachable.index) - set(eligible_cohorts))[-6:])

## Line comparison — earliest vs. most recent eligible cohorts

In [ ]:
eligible_sorted = sorted(eligible_cohorts)
sample_cohorts = eligible_sorted[:4] + eligible_sorted[-4:]
line_pivot = pivot.loc[sample_cohorts].T  # cycle as index, cohort as columns

fig, ax = plotting.plot_cohort_retention_lines(
    line_pivot, "Retention curves: earliest vs. most recent eligible cohorts",
    save_path="../figures/02_cohort_lines.png",
)

## Is the early-vs-late difference statistically significant?
Two-proportion z-test on cycle-6 retention: earliest quarter of eligible cohorts vs. most recent quarter.

In [ ]:
n_early = max(len(eligible_sorted) // 4, 1)
early_group, late_group = eligible_sorted[:n_early], eligible_sorted[-n_early:]

merged = periods.merge(members_cohort[["msno", "cohort_month"]], on="msno")
resolved = merged[merged["outcome"] != "censored"].copy()
resolved["cycle"] = resolved["period_seq"] + 1
at_cycle = resolved[resolved["cycle"] == COMPARE_CYCLE]

early_mask = at_cycle["cohort_month"].isin(early_group)
late_mask = at_cycle["cohort_month"].isin(late_group)

counts = [int(at_cycle.loc[early_mask, "renewed"].sum()), int(at_cycle.loc[late_mask, "renewed"].sum())]
nobs = [int(early_mask.sum()), int(late_mask.sum())]

z_stat, p_value = proportions_ztest(counts, nobs)
rate_early, rate_late = counts[0] / nobs[0], counts[1] / nobs[1]

print(f"Cycle {COMPARE_CYCLE} retention — earliest cohorts: {rate_early:.3f} (n={nobs[0]:,}), most recent eligible cohorts: {rate_late:.3f} (n={nobs[1]:,})")
print(f"z = {z_stat:.2f}, p = {p_value:.4f}")

If `p_value` is below 0.05, the retention difference between earliest and most recent eligible cohorts at cycle 6 is unlikely to be noise — report the direction and effect size, not just significance. If it's not significant, that's a legitimate finding too: cohort quality at this checkpoint has been roughly stable, and that should be reported as-is rather than reached past.

## Voluntary cancellation vs. silent lapse
Different problems, different fixes: voluntary cancel is a decision the business had a chance to intervene on; silent lapse often has no warning signal at all.

In [ ]:
outcome_counts = cohorts.cohort_outcome_counts(periods, members_cohort)
outcome_counts["voluntary_share"] = outcome_counts["voluntary_cancel"] / (
    outcome_counts["voluntary_cancel"] + outcome_counts["lapsed_no_renewal"]
)
print(outcome_counts.tail(12))

In [ ]:
table = outcome_counts.loc[eligible_cohorts, ["voluntary_cancel", "lapsed_no_renewal"]]
chi2, p, dof, _ = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.1f}, dof = {dof}, p = {p:.4g}")
print("Tests whether the MIX of voluntary-cancel vs. silent-lapse shifts across cohorts (not whether the total churn rate does).")

## Revenue retained vs. lost, by cohort

In [ ]:
revenue = cohorts.cohort_revenue_summary(periods, members_cohort)
recent_cohorts = eligible_cohorts[-12:] if len(eligible_cohorts) >= 12 else eligible_cohorts
revenue_recent = revenue.loc[recent_cohorts]

fig, ax = plotting.plot_revenue_outcome_bars(
    revenue_recent, "Revenue retained vs. lost, most recent eligible cohorts",
    save_path="../figures/02_revenue_outcome.png",
)

lost = revenue_recent[["voluntary_cancel", "lapsed_no_renewal"]].sum(axis=1)
retained = revenue_recent["renewed"]
total = retained.sum() + lost.sum()
print(f"Across these cohorts: NT${retained.sum():,.0f} retained vs NT${lost.sum():,.0f} lost ({lost.sum()/total:.1%} of resolved revenue).")
print(f"Of the lost revenue: {revenue_recent['voluntary_cancel'].sum()/lost.sum():.1%} voluntary cancellation, {revenue_recent['lapsed_no_renewal'].sum()/lost.sum():.1%} silent lapse.")

## Section summary

Fill in against the actual run output above:
- Cohort trend direction at cycle 6, and whether the z-test found it significant.
- Whether lost revenue skews toward voluntary cancellation (addressable with retention offers /
  win-back campaigns) or silent lapse (needs a different lever — renewal reminders, grace
  periods, auto-renew defaults).
- This feeds directly into the prioritization in the README's recommendation memo.